CARGAR DATASET

In [16]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np

RUTA_ENTRADA = "/content/drive/MyDrive/dataset_completo_tfm.csv"

df = pd.read_csv(RUTA_ENTRADA, low_memory=False)

print(f"Filas: {len(df):,}")
print(f"Columnas: {len(df.columns)}")
display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Filas: 96,474
Columnas: 94


,ID_FILA_SITIO,PK_RELACIONAL,NOMBRE_RECURSO,BARRIO_RECURSO,CATEGORIA_RECURSO,USUARIO,FECHA,IDIOMA,TEXTO_RESENA,CALIFICACION,...,FAX,EMAIL,TIPO,carpeta_origen,fichero_origen,PDF,HORARIO_es_sintetico,EQUIPAMIENTO_es_sintetico,DESCRIPCION_es_sintetico,ACCESIBILIDAD_es_sintetico
0,0,10000097.0,"GRACIAS, PADRE",JUSTICIA,HOSTELERÍA,Ildefonso Trinidad Martinez Corominas (España),2025-04-08,es,"Pésima experiencia, no pienso volver jamás. El...",1,...,NaN,NaN,NaN,NaN,NaN,NaN,True,True,True,True
1,0,10000097.0,"GRACIAS, PADRE",JUSTICIA,HOSTELERÍA,Andrew Davis (USA),2025-12-04,en,"Everything was perfect, exceeded all my expect...",5,...,NaN,NaN,NaN,NaN,NaN,NaN,True,True,True,True
2,0,10000097.0,"GRACIAS, PADRE",JUSTICIA,HOSTELERÍA,Valerio de Manzano (España),2025-01-01,es,"Decepcionante, esperaba bastante más por el pr...",1,...,NaN,NaN,NaN,NaN,NaN,NaN,True,True,True,True
3,1,10000150.0,CAFE LOS ARCOS BAR,SOL,HOSTELERÍA,Steven Ray (USA),2025-04-08,en,"An outstanding experience, I will definitely c...",4,...,NaN,NaN,NaN,NaN,NaN,NaN,True,True,True,True
4,1,10000150.0,CAFE LOS ARCOS BAR,SOL,HOSTELERÍA,Wera Herrmann (Alemania),2025-10-01,de,"Wirklich empfehlenswert, jedes Detail war groß...",5,...,NaN,NaN,NaN,NaN,NaN,NaN,True,True,True,True


REVISAR ESTRUCTURA

In [17]:
print("COLUMNAS DEL DATASET")
print(df.columns.tolist())

print("\nTIPOS DE DATOS")
display(df.dtypes.to_frame("tipo"))

COLUMNAS DEL DATASET
['ID_FILA_SITIO', 'PK_RELACIONAL', 'NOMBRE_RECURSO', 'BARRIO_RECURSO', 'CATEGORIA_RECURSO', 'USUARIO', 'FECHA', 'IDIOMA', 'TEXTO_RESENA', 'CALIFICACION', 'DATASET_SINTETICO', 'id_local', 'COD-DISTRITO', 'DISTRITO', 'id_barrio_local', 'BARRIO', 'COD-BARRIO', 'id_seccion_censal_local', 'desc_seccion_censal_local', 'COORDENADA-X', 'COORDENADA-Y', 'id_tipo_acceso_local', 'desc_tipo_acceso_local', 'id_situacion_local', 'desc_situacion_local', 'id_vial_edificio', 'clase_vial_edificio', 'desc_vial_edificio', 'id_ndp_edificio', 'id_clase_ndp_edificio', 'nom_edificio', 'num_edificio', 'cal_edificio', 'secuencial_local_PC', 'id_vial_acceso', 'CLASE-VIAL', 'NOMBRE-VIA', 'id_ndp_acceso', 'id_clase_ndp_acceso', 'nom_acceso', 'num_acceso', 'cal_acceso', 'coordenada_x_agrupacion', 'coordenada_y_agrupacion', 'id_agrupacion', 'nombre_agrupacion', 'id_tipo_agrup', 'desc_tipo_agrup', 'id_planta_agrupado', 'id_local_agrupado', 'NOMBRE', 'id_seccion', 'categoria', 'id_division', 'desc_

,tipo
ID_FILA_SITIO,int64
PK_RELACIONAL,object
NOMBRE_RECURSO,object
BARRIO_RECURSO,object
CATEGORIA_RECURSO,object
...,...
PDF,object
HORARIO_es_sintetico,bool
EQUIPAMIENTO_es_sintetico,bool
DESCRIPCION_es_sintetico,bool


ANALIZAR VALORES VACÍOS

In [18]:
informe_nulos = pd.DataFrame({
    "valores_nulos": df.isna().sum(),
    "porcentaje_nulos": (df.isna().mean() * 100).round(2)
})

informe_nulos = informe_nulos.sort_values(
    "porcentaje_nulos",
    ascending=False
)

display(informe_nulos)

,valores_nulos,porcentaje_nulos
PLANTA,96471,100.00
ESCALERAS,96474,100.00
FAX,96447,99.97
PUERTA,96438,99.96
ORIENTACION,96174,99.69
...,...,...
EQUIPAMIENTO,0,0.00
HORARIO_es_sintetico,0,0.00
EQUIPAMIENTO_es_sintetico,0,0.00
DESCRIPCION_es_sintetico,0,0.00


COMPROBAR DUPLICADOS

In [21]:
numero_duplicados = df.duplicated().sum()

print("Filas completamente duplicadas:", numero_duplicados)

if numero_duplicados > 0:
    display(df[df.duplicated(keep=False)].head(20))

if "TEXTO_RESENA" in df.columns:
    textos_repetidos = df["TEXTO_RESENA"].duplicated().sum()
    print("Textos de reseña repetidos:", textos_repetidos)
else:
    print("No se encuentra la columna TEXTO_RESENA")

Filas completamente duplicadas: 0
Textos de reseña repetidos: 96103


VALIDAR CALIFICACIONES

In [22]:
df["CALIFICACION"] = pd.to_numeric(
    df["CALIFICACION"],
    errors="coerce"
)

calificaciones_invalidas = df[
    df["CALIFICACION"].isna()
    | ~df["CALIFICACION"].between(1, 5)
]

print(
    "Calificaciones inválidas:",
    len(calificaciones_invalidas)
)

print("\nDistribución de calificaciones:")
display(
    df["CALIFICACION"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("cantidad")
)

print("Calificación media:", round(df["CALIFICACION"].mean(), 2))

Calificaciones inválidas: 0

Distribución de calificaciones:


,cantidad
CALIFICACION,
1,8956
2,13662
3,19815
4,27887
5,26154


Calificación media: 3.5


VALIDAR IDENTIFICADORES Y CAMPOS PRINCIPALES

In [24]:
columnas_principales = [
    "PK_RELACIONAL",
    "NOMBRE_RECURSO",
    "BARRIO_RECURSO",
    "CATEGORIA_RECURSO",
    "TEXTO_RESENA",
    "CALIFICACION"
]

resultado_principales = []

for columna in columnas_principales:
    if columna in df.columns:
        resultado_principales.append({
            "columna": columna,
            "existe": "Sí",
            "valores_vacios": df[columna].isna().sum(),
            "porcentaje_vacio": round(df[columna].isna().mean() * 100, 2)
        })
    else:
        resultado_principales.append({
            "columna": columna,
            "existe": "No",
            "valores_vacios": None,
            "porcentaje_vacio": None
        })

display(pd.DataFrame(resultado_principales))


#Comprobar cuántos recursos diferentes existen y cuántas opiniones tiene cada uno:
if "PK_RELACIONAL" in df.columns:
    print(
        "Número de recursos diferentes:",
        df["PK_RELACIONAL"].nunique()
    )

    opiniones_por_recurso = (
        df.groupby("PK_RELACIONAL")
        .size()
        .describe()
    )

    display(opiniones_por_recurso.to_frame("resultado"))

,columna,existe,valores_vacios,porcentaje_vacio
0,PK_RELACIONAL,Sí,0,0.00
1,NOMBRE_RECURSO,Sí,57,0.06
2,BARRIO_RECURSO,Sí,198,0.21
3,CATEGORIA_RECURSO,Sí,0,0.00
4,TEXTO_RESENA,Sí,0,0.00
5,CALIFICACION,Sí,0,0.00


Número de recursos diferentes: 30725


,resultado
count,30725.000000
mean,3.139919
std,0.686353
min,3.000000
25%,3.000000
50%,3.000000
75%,3.000000
max,15.000000


VALIDAR FECHAS, IDIOMAS Y ORIGEN SINTÉTICO

In [25]:
if "FECHA" in df.columns:
    df["FECHA"] = pd.to_datetime(
        df["FECHA"],
        errors="coerce"
    )

    print("Fechas inválidas:", df["FECHA"].isna().sum())
    print("Fecha mínima:", df["FECHA"].min())
    print("Fecha máxima:", df["FECHA"].max())

if "IDIOMA" in df.columns:
    print("\nDistribución de idiomas:")
    display(df["IDIOMA"].value_counts(dropna=False).to_frame("cantidad"))

if "DATASET_SINTETICO" in df.columns:
    print("\nIdentificación de datos sintéticos:")
    display(
        df["DATASET_SINTETICO"]
        .value_counts(dropna=False)
        .to_frame("cantidad")
    )

Fechas inválidas: 0
Fecha mínima: 2025-01-01 00:00:00
Fecha máxima: 2025-12-28 00:00:00

Distribución de idiomas:


,cantidad
IDIOMA,
es,52929
en,21295
de,11688
fr,10562



Identificación de datos sintéticos:


,cantidad
DATASET_SINTETICO,
True,96474


REVISAR CATEGORÍAS Y BARRIOS

In [26]:
if "CATEGORIA_RECURSO" in df.columns:
    print("Categorías:")
    display(
        df["CATEGORIA_RECURSO"]
        .value_counts(dropna=False)
        .to_frame("cantidad")
    )

if "BARRIO_RECURSO" in df.columns:
    print("\nNúmero de barrios diferentes:")
    print(df["BARRIO_RECURSO"].nunique())

    display(
        df["BARRIO_RECURSO"]
        .value_counts(dropna=False)
        .head(30)
        .to_frame("cantidad")
    )

Categorías:


,cantidad
CATEGORIA_RECURSO,
HOSTELERÍA,92211
instalacion_deportiva,1836
monumento,1038
parque,633
museo,213
piscina,192
mercado,135
sala_ocio,117
mercadillo,99



Número de barrios diferentes:
147


,cantidad
BARRIO_RECURSO,
EMBAJADORES,5550
UNIVERSIDAD,4560
SOL,4170
PALACIO,4026
CORTES,3573
JUSTICIA,2751
CUATRO CAMINOS,1572
GOYA,1539
RECOLETOS,1413


GENERAR EL INFORME DE CALIDAD

In [27]:
resumen_calidad = pd.DataFrame({
    "control": [
        "Registros totales",
        "Filas duplicadas",
        "Identificadores vacíos",
        "Calificaciones inválidas",
        "Fechas inválidas",
        "Textos repetidos"
    ],
    "resultado": [
        len(df),
        df.duplicated().sum(),
        df["PK_RELACIONAL"].isna().sum(),
        len(calificaciones_invalidas),
        df["FECHA"].isna().sum(),
        df["TEXTO_RESENA"].duplicated().sum()
    ]
})

display(resumen_calidad)

,control,resultado
0,Registros totales,96474
1,Filas duplicadas,0
2,Identificadores vacíos,0
3,Calificaciones inválidas,0
4,Fechas inválidas,0
5,Textos repetidos,96103
